**© Copyright AIDENTIFY. All rights reserved.**

본 자료는 **멀티캠퍼스 LLM 파인튜닝 과정** 수강생을 위해 제작되었으며, 강의 목적으로만 사용 가능합니다.  
무단 복제, 배포, 상업적 이용을 금지합니다.

---

# 🎯 실습 환경 점검 노트북
이 노트북을 실행하여 LLM 파인튜닝 실습에 필요한 환경이 올바르게 설정되었는지 확인합니다.

## 1️⃣ 시스템 정보 확인

In [ ]:
import platform
import sys

print(f"OS: {platform.system()} {platform.release()}")
print(f"Python: {sys.version}")
print(f"Python 경로: {sys.executable}")

## 2️⃣ GPU 확인

In [ ]:
import torch

print(f"PyTorch 버전: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA 버전: {torch.version.cuda}")
    print(f"GPU 이름: {torch.cuda.get_device_name(0)}")
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU VRAM: {total_mem:.1f} GB")
    
    if total_mem < 8:
        print("\n⚠️ VRAM이 8GB 미만입니다. 일부 실습에 제한이 있을 수 있습니다.")
    elif total_mem < 16:
        print("\n✅ RTX 4060급 GPU입니다. QLoRA로 7B 모델까지 학습 가능합니다.")
    else:
        print(f"\n✅ VRAM {total_mem:.0f}GB - 대부분의 실습이 가능합니다.")
else:
    print("\n⚠️ GPU를 사용할 수 없습니다. CPU 모드로 진행하거나 GPU 클라우드를 사용하세요.")

## 3️⃣ 필수 패키지 확인

In [ ]:
required_packages = {
    "torch": "PyTorch (딥러닝 프레임워크)",
    "transformers": "Transformers (모델 로딩/추론)",
    "peft": "PEFT (LoRA/QLoRA 파인튜닝)",
    "trl": "TRL (SFT/DPO/GRPO 학습)",
    "datasets": "Datasets (데이터셋 처리)",
    "accelerate": "Accelerate (분산 학습)",
    "bitsandbytes": "BitsAndBytes (양자화)",
}

optional_packages = {
    "openai": "OpenAI API",
    "langchain": "LangChain (RAG 프레임워크)",
    "chromadb": "ChromaDB (벡터 DB)",
    "tiktoken": "tiktoken (토큰화)",
    "ragas": "RAGAS (RAG 평가)",
    "streamlit": "Streamlit (웹앱)",
    "matplotlib": "Matplotlib (시각화)",
}

print("=== 필수 패키지 ===")
all_ok = True
for pkg, desc in required_packages.items():
    try:
        mod = __import__(pkg)
        ver = getattr(mod, "__version__", "?")
        print(f"  ✅ {desc}: {ver}")
    except ImportError:
        print(f"  ❌ {desc}: 설치 필요 (pip install {pkg})")
        all_ok = False

print("\n=== 선택 패키지 ===")
for pkg, desc in optional_packages.items():
    try:
        mod = __import__(pkg)
        ver = getattr(mod, "__version__", "?")
        print(f"  ✅ {desc}: {ver}")
    except ImportError:
        print(f"  ⬜ {desc}: 미설치 (해당 파트에서 필요 시 설치)")

if all_ok:
    print("\n🎉 필수 패키지가 모두 설치되어 있습니다!")
else:
    print("\n⚠️ 누락된 필수 패키지를 설치해주세요: pip install -r requirements.txt")

## 4️⃣ API 키 확인

In [ ]:
import os

# .env 파일이 있으면 로드
try:
    from dotenv import load_dotenv
    load_dotenv()
    print(".env 파일 로드 완료")
except ImportError:
    print("python-dotenv 미설치 (pip install python-dotenv)")

# OpenAI API 키 확인
openai_key = os.getenv("OPENAI_API_KEY", "")
if openai_key and openai_key != "sk-your-api-key-here":
    print(f"✅ OPENAI_API_KEY 설정됨 (sk-...{openai_key[-4:]})")
else:
    print("⬜ OPENAI_API_KEY 미설정 (Part 1, 3, 4, 5에서 필요)")

# HuggingFace 토큰 확인
hf_token = os.getenv("HF_TOKEN", "")
if hf_token and hf_token != "hf_your-token-here":
    print(f"✅ HF_TOKEN 설정됨 (hf_...{hf_token[-4:]})")
else:
    print("⬜ HF_TOKEN 미설정 (공개 모델은 토큰 없이 다운로드 가능)")

## 5️⃣ 디스크 공간 확인

In [ ]:
import shutil

total, used, free = shutil.disk_usage("/")
print(f"전체: {total / 1024**3:.1f} GB")
print(f"사용: {used / 1024**3:.1f} GB")
print(f"여유: {free / 1024**3:.1f} GB")

if free / 1024**3 < 20:
    print("\n⚠️ 디스크 여유 공간이 20GB 미만입니다. 모델 다운로드에 문제가 있을 수 있습니다.")
else:
    print("\n✅ 디스크 공간 충분")

# HuggingFace 캐시 경로
hf_home = os.getenv("HF_HOME", os.path.expanduser("~/.cache/huggingface"))
print(f"\nHuggingFace 캐시 경로: {hf_home}")

## 6️⃣ GPU 메모리 모니터링 테스트

In [ ]:
import sys
sys.path.append('..')

try:
    from utils.gpu_monitor import print_gpu_memory, clear_gpu_memory
    print_gpu_memory("초기 상태")
    print("✅ GPU 모니터링 유틸리티 정상 작동")
except Exception as e:
    print(f"⚠️ GPU 모니터링 유틸리티 로드 실패: {e}")
    print("직접 GPU 메모리 확인:")
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"  GPU 메모리: {allocated:.1f}GB / {total:.1f}GB")

## 🎉 환경 점검 완료!

모든 항목이 ✅이면 실습 준비가 완료된 것입니다.

### 문제 해결
- 패키지 설치: `pip install -r requirements.txt`
- GPU 미인식: NVIDIA 드라이버 및 CUDA 설치 확인
- API 키: `.env.example`을 `.env`로 복사하고 키 입력